# Exhaust Manifolds

These are 3D printable exhaust manifolds which connects a muffler tube to exhaust tips. There are 2 manifolds, one driver and one passenger. The manifolds are further subdivided into right and left sides so that inner supports can be accessed after 3D printing. Each side includes a guide so that the subassemblies can be aligned.

The outer diameter of the tubing is 2.5", and the inlets are connected to midpipes with slide on clamps. The exhaust tips have their own clamps which fit over the tubing. 

## Creating the manifolds

The data looks correct, so now we need to build the manifold in cadquery. To support splitting the design into multiple 3D printing meshes, the function allows variable sweep and trim profiles. 

In [ ]:
%load_ext autoreload
%autoreload 2

from build import Builder
import cadquery as cq
from jupyter_cadquery import *
import logging

logging.basicConfig(
    filename="out.txt",
    level=logging.DEBUG,
    filemode='w'
)

builder = Builder()

driver_wire, passenger_wire = builder.build_wire("driver"), builder.build_wire("passenger")
show(driver_wire, passenger_wire)

Build the manifolds so we can perform some basic verification of the shapes.

In [ ]:
builder = Builder()
driver_manifold, passenger_manifold = builder.build_manifold("driver"), builder.build_manifold("passenger")

assy = cq.Assembly()
assy.add(driver_manifold, name="driver_manifold", color="red")
assy.add(passenger_manifold, name="passenger_manifold", color="green")
show(assy)

Split the manifolds so inner supports can be removed after 3D printing, then ensure the manifold diameters were correct.

In [ ]:
builder = Builder()

assy = cq.Assembly()
assy.add(builder.build_manifold_half("driver"), name="driver_left", color="red")
assy.add(builder.build_manifold_half("driver", right = True), name="driver_right", color="green")
assy.add(builder.build_manifold_half("passenger"), name="passenger_left", color="purple")
assy.add(builder.build_manifold_half("passenger", right = True), name="passenger_right", color="yellow")
show(assy)

Build guidelines for the split manifolds.

In [ ]:
builder = Builder()

assy = cq.Assembly()
assy.add(builder.build_manifold_half("driver"), name="driver_left", color="red")
assy.add(builder.build_manifold_half("driver", right=True), name="driver_right", color="green")
assy.add(builder.build_manifold_half("passenger"), name="passenger_left", color="purple")
assy.add(builder.build_manifold_half("passenger", right=True), name="passenger_right", color="yellow")
assy.add(builder.build_guide("driver"), name="driver_guide_left", color="cyan")
assy.add(builder.build_guide("driver", right=True), name="driver_guide_right", color="orange")
assy.add(builder.build_guide("passenger"), name="passenger_guide_left", color="blue")
assy.add(builder.build_guide("passenger", right=True), name="passenger_guide_right", color="orange")
show(assy)

Build all parts, then the parts to ensure ensure no manifold volume is missing from individual parts. 

In [ ]:
builder = Builder()

driver_manifold = builder.build_manifold("driver")
passenger_manifold = builder.build_manifold("passenger")
driver_left = builder.build_part("driver")
driver_right = builder.build_part("driver", right=True)
passenger_left = builder.build_part("passenger")
passenger_right = builder.build_part("passenger", right=True)

assy = cq.Assembly()
assy.add(driver_left, name="driver_left", color="red")
assy.add(driver_right, name="driver_right", color="green")
assy.add(passenger_left, name="passenger_left", color="purple")
assy.add(passenger_right, name="passenger_right", color="yellow")
show(assy)

As a sanity check, show that we can recombine the parts back into the original manifold shape.

In [ ]:
builder = Builder()

for name in ["driver", "passenger"]:
    error_pct = builder.calc_part_error(name)
    print(f"{name} error: {round(error_pct)}%")

driver_manifold, driver_manifold_from_parts = builder.build_back_manifold("driver")
passenger_manifold, passenger_manifold_from_parts = builder.build_back_manifold("passenger")

assy = cq.Assembly()
assy.add(driver_manifold, name="driver_manifold", color="red")
assy.add(driver_manifold_from_parts, name="driver_manifold_from_parts", color="blue")
assy.add(passenger_manifold, name="passenger_manifold", color="green")
assy.add(passenger_manifold_from_parts, name="passenger_manifold_from_parts", color="yellow")
show(assy)